# Juliet Utility Router 학습

이 노트북은 위에서 아래로 한 번 실행하면 됩니다. `.env`에는 `OPENROUTER_API_KEY`만 있으면 됩니다. 각 Juliet case의 모든 candidate와 E1/E3/E4/E5/E6 작업을 하나의 batch prompt에 넣으므로 **case 1건당 물리 API 호출은 최대 1회**입니다. 결과는 case 단위로 저장되며 중단 후 다시 실행하면 완료된 case를 건너뜁니다.

In [1]:
from pathlib import Path

EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
TRAIN_CASE_LIMIT = 5  # 0 = frozen train split 전체
DEV_CASE_LIMIT = 2    # 0 = frozen dev split 전체
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
TARGET_TRUTH_RECALL = 0.95


In [2]:
import json, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.workflow import (resolve_models, plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, train_utility_router)
from model_evaluation.adapters.llm_security import expert_assignments
config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
models = resolve_models(ENV_FILE)
RUN_DIR = EVAL_ROOT / 'work' / 'router_training'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_training'
ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print('Physical model:', models[0])

Physical model: deepseek/deepseek-v4-flash-0731


## 1. Frozen train/dev case 생성

In [3]:
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases',
    splits=('train', 'dev'), limits={'train': TRAIN_CASE_LIMIT, 'dev': DEV_CASE_LIMIT},
    progress=print,
)
print(json.dumps(materialization, ensure_ascii=False, indent=2))

{
  "mapping_hash": "57c2452af46fbd70f7e8e7416c1c5f76890f97276515e4a0cbd03acb4c56c719",
  "schema_version": "juliet-eval-v1",
  "seed": 2026,
  "splits": {
    "dev": {
      "attempted_cases": 2,
      "available_indexed_cases": 8324,
      "cases": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\cases\\cases_dev.jsonl",
      "cwe_distribution": {
        "CWE-121": 1,
        "CWE-122": 1
      },
      "expert_distribution": {
        "memory_bounds": 2
      },
      "failure_count": 0,
      "failures": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\cases\\failures_dev.jsonl",
      "materialized_cases": 2,
      "requested_limit": 2
    },
    "train": {
      "attempted_cases": 5,
      "available_indexed_cases": 38563,
      "cases": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\cases\\cases_train.jsonl",
      "cwe_distribution": {
        "CWE-121": 1,
        "CWE-122": 1,
        "CWE-123": 1,
        "CWE-124": 1,
        "CWE-126": 

## 2. Semantic Analyzer candidate 캐시

In [4]:
for split in ('train', 'dev'):
    summary = cache_candidates(
        RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
    )
    print(split, json.dumps(summary, ensure_ascii=False, indent=2))

train {
  "analysis_failure_count": 0,
  "candidate_cache": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\candidates\\candidates_train.jsonl",
  "candidate_count": 22,
  "case_count": 5,
  "cases": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\cases\\cases_train.jsonl",
  "cases_mtime_ns": 1787302987457825200,
  "cases_size": 43433,
  "elapsed_seconds": 0.08978279999791994,
  "feature_schema": "semantic-cwe-v2",
  "max_source_bytes": null,
  "parse_timeout_ms": 30000
}
dev {
  "analysis_failure_count": 0,
  "candidate_cache": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\candidates\\candidates_dev.jsonl",
  "candidate_count": 9,
  "case_count": 2,
  "cases": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\cases\\cases_dev.jsonl",
  "cases_mtime_ns": 1787302987506366300,
  "cases_size": 16236,
  "elapsed_seconds": 0.02704470000026049,
  "feature_schema": "semantic-cwe-v2",
  "max_source_bytes": null,
  "parse_timeout_ms": 30000
}


## 3. 호출 계획 — logical Expert 5개를 case당 API 1회로 batch

In [5]:
plans = {}
for split in ('train', 'dev'):
    plans[split] = plan_outcome_matrix(
        cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
print(json.dumps(plans, ensure_ascii=False, indent=2))

{
  "train": {
    "selection_manifest": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\selections\\selected_train.jsonl",
    "selected_candidate_count": 10,
    "positive_candidate_count": 5,
    "hard_negative_candidate_count": 5,
    "model": "deepseek/deepseek-v4-flash-0731",
    "assignment_count": 5,
    "expected_physical_api_requests": 5,
    "completed_physical_api_requests": 0,
    "remaining_physical_api_requests": 5,
    "expected_logical_expert_outcomes": 50,
    "completed_logical_expert_outcomes": 0,
    "unexpected_existing_rows": 0,
    "outcome_path": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\outcomes\\outcomes_train.jsonl"
  },
  "dev": {
    "selection_manifest": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\selections\\selected_dev.jsonl",
    "selected_candidate_count": 4,
    "positive_candidate_count": 2,
    "hard_negative_candidate_count": 2,
    "model": "deepseek/deepseek-v4-flash-0731",
    "assignment_count": 5

## 4. Batch outcome 수집

추가 스위치는 없습니다. API key가 있으면 바로 실행합니다. 각 case의 최대 4개 candidate × 5 Expert를 한 요청으로 처리합니다.

In [6]:
collection_reports = {}
for split in ('train', 'dev'):
    report = collect_outcome_matrix(
        env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        ledger_path=RUN_DIR / 'ledgers' / f'{split}_api_ledger.jsonl',
        model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
    collection_reports[split] = report
    print(split, json.dumps(report, ensure_ascii=False, indent=2))
    if report['status'] != 'complete':
        print('중단 지점이 저장되었습니다. 원인을 해결한 뒤 이 셀을 다시 실행하면 이어서 진행합니다.')
        break

batched outcomes: 1/1 completed; physical requests this run=1
batched outcomes: 2/2 completed; physical requests this run=2
batched outcomes: 3/3 completed; physical requests this run=3
batched outcomes: 4/4 completed; physical requests this run=4
batched outcomes: 5/5 completed; physical requests this run=5
train {
  "status": "complete",
  "stop_reason": null,
  "model": "deepseek/deepseek-v4-flash-0731",
  "cases_seen": 5,
  "completed_cases": 5,
  "new_cases": 5,
  "physical_requests_this_run": 5,
  "actual_cost_usd_this_run": 0.003314695,
  "request_contract": "at most one physical detection request per case",
  "outcome_path": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\outcomes\\outcomes_train.jsonl",
  "ledger_path": "D:\\llm-security\\Model_Evaluation\\work\\router_training\\ledgers\\train_api_ledger.jsonl"
}
batched outcomes: 1/1 completed; physical requests this run=1
dev {
  "status": "stopped_on_error_resumable",
  "stop_reason": "juliet-sard-ef062714d0837e

## 5. 완전성 검사 후 Utility Router + Escalation Gate 학습

In [7]:
expected_ids = [item.assignment_id for item in expert_assignments(models)]
outcome_files = {s: RUN_DIR / 'outcomes' / f'outcomes_{s}.jsonl' for s in ('train', 'dev')}
audits = {s: audit_outcome_matrix(p, expected_assignment_ids=expected_ids, selection_manifest=RUN_DIR / 'selections' / f'selected_{s}.jsonl') if p.exists() else {'complete': False, 'reason': 'missing'} for s, p in outcome_files.items()}
print(json.dumps(audits, ensure_ascii=False, indent=2))
if all(item['complete'] for item in audits.values()):
    training_report = train_utility_router(
        train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
        train_selection_manifest=RUN_DIR / 'selections' / 'selected_train.jsonl',
        dev_selection_manifest=RUN_DIR / 'selections' / 'selected_dev.jsonl',
        artifact_path=ARTIFACT, report_path=RESULT_DIR / 'training_report.json',
        model_ids=models, seed=config.seed, target_truth_recall=TARGET_TRUTH_RECALL,
    )
    print(json.dumps(training_report, ensure_ascii=False, indent=2))
else:
    print('outcome 수집이 아직 끝나지 않았습니다. 4번 셀을 다시 실행하면 이어서 진행합니다.')

{
  "train": {
    "row_count": 50,
    "candidate_group_count": 10,
    "expected_assignment_count": 5,
    "duplicate_row_count": 0,
    "incomplete_candidate_group_count": 0,
    "incomplete_preview": [],
    "expected_candidate_group_count": 10,
    "missing_candidate_group_count": 0,
    "missing_preview": [],
    "unexpected_candidate_group_count": 0,
    "complete": true
  },
  "dev": {
    "row_count": 10,
    "candidate_group_count": 2,
    "expected_assignment_count": 5,
    "duplicate_row_count": 0,
    "incomplete_candidate_group_count": 0,
    "incomplete_preview": [],
    "expected_candidate_group_count": 4,
    "missing_candidate_group_count": 2,
    "missing_preview": [
      "juliet-sard-ef062714d0837ee4a418/C-6fdf0a20a8ea228d",
      "juliet-sard-ef062714d0837ee4a418/C-f15638a8641b7608"
    ],
    "unexpected_candidate_group_count": 0,
    "complete": false
  }
}
outcome 수집이 아직 끝나지 않았습니다. 4번 셀을 다시 실행하면 이어서 진행합니다.
